In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention
    
    Computes the attention weights using the formula:
        Attention(Q, K, V) = softmax((Q * K^T) / sqrt(d_model))
    """
    def __init__(self):
        super(ScaledDotProductAttention, self).__init__()

    def forward(self, query, key, value, mask=None):
        """
        Compute attention weights and output.

        Args:
            query: Query tensor of shape [batch_size, seq_len, d_model]
            key: Key tensor of shape [batch_size, seq_len, d_model]
            value: Value tensor of shape [batch_size, seq_len, d_model]
            mask: Optional mask tensor (same shape as attention scores)

        Returns:
            output: Attention output tensor [batch_size, seq_len, d_model]
            attention_weights: Attention weights [batch_size, seq_len, seq_len]
        """
        d_q = query.size()[-1]

        # Compute scaled dot-product attention scores - [batch_size, seq_len, seq_len]
        scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_q, dtype=torch.float32))

        # Apply mask (if provided)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # Apply softmax to get attention weights
        attention_weights = torch.softmax(scores, dim=-1)

        # Compute the final output  - - [batch_size, seq_len, d_model]
        output = torch.matmul(attention_weights, value)

        return output, attention_weights


In [3]:
sdpta = ScaledDotProductAttention()

batch_size, seq_len, d_model = 16, 20, 768

query = torch.randn(batch_size, seq_len, d_model)
key = torch.randn(batch_size, seq_len, d_model)
value = torch.randn(batch_size, seq_len, d_model)
output, scores = sdpta(query, key, value)

print(f"output size is {output.size()}")
print(f"score size is {scores.size()}")
print(f"score is {scores}")

output size is torch.Size([16, 20, 768])
score size is torch.Size([16, 20, 20])
score is tensor([[[0.1025, 0.0109, 0.0121,  ..., 0.0176, 0.0372, 0.0336],
         [0.2294, 0.0079, 0.1975,  ..., 0.0241, 0.0306, 0.0759],
         [0.0642, 0.0209, 0.0072,  ..., 0.1005, 0.0251, 0.0058],
         ...,
         [0.0024, 0.0248, 0.0481,  ..., 0.1708, 0.0032, 0.1055],
         [0.0258, 0.0898, 0.0569,  ..., 0.0154, 0.0429, 0.0056],
         [0.0860, 0.0328, 0.0168,  ..., 0.0298, 0.0106, 0.0274]],

        [[0.0065, 0.0083, 0.0473,  ..., 0.1729, 0.0086, 0.0222],
         [0.0090, 0.1203, 0.0069,  ..., 0.0474, 0.0356, 0.0632],
         [0.0451, 0.0474, 0.0394,  ..., 0.0639, 0.0369, 0.0249],
         ...,
         [0.0552, 0.2120, 0.1161,  ..., 0.0320, 0.0154, 0.0045],
         [0.0266, 0.0568, 0.0484,  ..., 0.0835, 0.0230, 0.0352],
         [0.0675, 0.0475, 0.1669,  ..., 0.0374, 0.0332, 0.0067]],

        [[0.0700, 0.1031, 0.0046,  ..., 0.0071, 0.0101, 0.0189],
         [0.0155, 0.0261, 0.0877, 

In [4]:
sum(scores[0][0])

tensor(1.)

In [5]:
# generate mask
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
print(mask)
mask.size()

tensor([[[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0.,
          0.,

torch.Size([1, 20, 20])

In [6]:
output, scores = sdpta(query,key,value, mask=mask)

print(f"output size is {output.size()}")
print(f"score size is {scores.size()}")
print(f"score is {scores[0]}")

output size is torch.Size([16, 20, 768])
score size is torch.Size([16, 20, 20])
score is tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.9666, 0.0334, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.6955, 0.2264, 0.0781, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.4109, 0.0833, 0.2459, 0.2599, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.0642, 0.0943, 0.4243, 0.1280, 0.2893, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.000

In [7]:
class MultiheadAttention(nn.Module):
    """
    Multi-Head Attention Module

    Splits the input into multiple heads, performs Scaled Dot-Product Attention 
    on each head independently, and then concatenates the results followed by a 
    linear projection.
    """
    
    def __init__(self, d_model, num_heads):
        """
        Initialize the Multi-Head Attention layer.

        Args:
            d_model (int): The dimensionality of the model.
            num_heads (int): The number of attention heads.
        """
        super(MultiheadAttention, self).__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        
        # Ensure the model dimension is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.head_dim = self.d_model // self.num_heads
        
        # Linear projections for query, key, and value
        self.query_proj = nn.Linear(d_model, d_model)
        self.key_proj = nn.Linear(d_model, d_model)
        self.val_proj = nn.Linear(d_model, d_model)
        
        # Output projection after concatenation
        self.out_proj = nn.Linear(d_model, d_model)
        
        # Scaled Dot-Product Attention module
        self.attention = ScaledDotProductAttention()
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, head_dim) 
        and transpose to shape [batch_size, num_heads, seq_len, head_dim].

        Args:
            x: Tensor of shape [batch_size, seq_len, d_model]

        Returns:
            Tensor of shape [batch_size, num_heads, seq_len, head_dim]
        """
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine multiple heads into a single tensor.

        Args:
            x: Tensor of shape [batch_size, num_heads, seq_len, head_dim]

        Returns:
            Tensor of shape [batch_size, seq_len, d_model]
        """
        batch_size, num_heads, seq_len, head_dim = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, num_heads * head_dim)
        
    def forward(self, x, mask=None):
        """
        Perform multi-head attention computation.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]
            mask: Optional mask tensor [batch_size, seq_len]

        Returns:
            output: Final output tensor [batch_size, seq_len, d_model]
            attention_weights: Attention weights [batch_size, num_heads, seq_len, seq_len]
        """
        batch_size, seq_len, d_model = x.size()
        
        # Linear projections
        query = self.query_proj(x)
        key = self.key_proj(x)
        value = self.val_proj(x)
        
        # Split into multiple heads
        query_split = self.split_heads(query)
        key_split = self.split_heads(key)
        value_split = self.split_heads(value)
        
        # Expand mask for all heads
        if mask is not None:
            mask = mask.unsqueeze(1)
        
        # Compute attention
        attention_out, attention_weights = self.attention(query_split, key_split, value_split, mask)
        # attention_out shape: [batch_size, num_heads, seq_len, head_dim]

        # Combine heads
        attention_out = self.combine_heads(attention_out)
        
        # Final linear projection
        output = self.out_proj(attention_out)
        
        return output, attention_weights

In [ ]:
class MaskedMultiHeadAttention(MultiheadAttention):
    """
    Masked Multi-Head Attention Module

    This layer is specifically used in the Transformer decoder’s self-attention block.
    It adds a future mask (causal mask) to prevent a position i from attending 
    to any future position j > i.
    """
    
    def __init__(self, d_model, num_heads):
        super(MaskedMultiHeadAttention, self).__init__(d_model, num_heads)
    
    def forward(self, x):
        """
        Compute masked multi-head self-attention.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]

        Returns:
            output: Masked attention output [batch_size, seq_len, d_model]
            attention_weights: Attention weights 
                               [batch_size, num_heads, seq_len, seq_len]
        """
        seq_len = x.size(1)
        
        # Create a lower-triangular (causal) mask: 1s on and below the diagonal, 0s above
        # Shape: [1, seq_len, seq_len]
        mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
        
        # Call the parent class forward method with the mask applied
        return super().forward(x, mask)

In [9]:


batch_size, seq_len, d_model = 16, 20, 768

mmha = MaskedMultiHeadAttention(d_model, num_heads=12)
x = torch.randn(batch_size, seq_len, d_model)

output, scores = mmha(x)


print(f"output size is {output.size()}")
print(f"score size is {scores.size()}")
print(f"score is {scores[0]}")

output size is torch.Size([16, 20, 768])
score size is torch.Size([16, 12, 20, 20])
score is tensor([[[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.5987, 0.4013, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.1732, 0.4059, 0.4209,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0704, 0.0496, 0.0805,  ..., 0.0428, 0.0000, 0.0000],
         [0.0563, 0.0532, 0.0438,  ..., 0.0462, 0.0454, 0.0000],
         [0.0311, 0.0471, 0.0617,  ..., 0.0407, 0.0449, 0.0457]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.3760, 0.6240, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.4445, 0.3002, 0.2554,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.0551, 0.0516, 0.0520,  ..., 0.0455, 0.0000, 0.0000],
         [0.0609, 0.0506, 0.0565,  ..., 0.0656, 0.0900, 0.0000],
         [0.0484, 0.0709, 0.0512,  ..., 0.0553, 0.0479, 0.0334]],

        [[1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.5057, 0.4943, 0.00

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional Encoding Module (supports dynamic sequence lengths)

    Adds sinusoidal positional information to token embeddings.
    This allows the model to capture the order of tokens in a sequence
    without relying on recurrence or convolution.
    """
    def __init__(self, d_model, max_len=5000):
        """
        Initialize the positional encoding matrix.

        Args:
            d_model (int): The dimensionality of the model embeddings.
            max_len (int): The maximum sequence length supported.
        """
        super().__init__()
        
        # Create a matrix of shape [max_len, d_model] to store positional encodings
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sine to even indices and cosine to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as a buffer so it's not treated as a learnable parameter
        # Shape: [1, max_len, d_model]
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        """
        Add positional encoding to the input embeddings.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]

        Returns:
            Tensor with positional encodings added [batch_size, seq_len, d_model]
        """
        # Get positional encodings dynamically based on the input sequence length
        position_emb = self.pe[:, :x.size(1)]
        return x + position_emb

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network (with GeLU activation)

    Applies two linear transformations with a GeLU activation in between,
    independently to each position in the sequence. Includes Layer Normalization 
    and residual connections for stability.
    """
    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        """
        Initialize the feed-forward network.

        Args:
            d_model (int): Dimensionality of the input and output.
            d_ff (int): Dimensionality of the inner feed-forward layer.
            dropout_rate (float): Dropout probability (currently optional).
        """
        super().__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.GELU()  # GeLU activation function
        # self.res_dropout = nn.Dropout(dropout_rate)  # Optional dropout

    def forward(self, x):
        """
        Forward pass of the position-wise feed-forward layer.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]

        Returns:
            Tensor of shape [batch_size, seq_len, d_model] after applying 
            feed-forward transformation and residual connection.
        """
        residual = x  # Residual connection
        output = self.layer_norm(x)  # Pre-layer normalization
        output = self.linear1(output)
        output = self.activation(output)
        output = self.linear2(output)
        
        # Optionally apply dropout before adding the residual
        # output = self.res_dropout(output)
        
        return output + residual

In [ ]:
# Define batch size, sequence length, and hidden size
batch_size, seq_len, hidden_size = 16, 10, 768

# Create a random input tensor simulating transformer embeddings
x = torch.randn(batch_size, seq_len, hidden_size)

print(f"Input size: {x.size()}")
print(f"First sample (before feed-forward):\n{x[0]}")

# Initialize the Position-wise Feed-Forward Network
pffn = PositionwiseFeedForward(hidden_size, hidden_size * 4)

# Pass the input through the feed-forward layer
output = pffn(x)

print(f"Output size: {output.size()}")
print(f"First sample (after feed-forward):\n{output[0]}")

intput size is torch.Size([16, 10, 768])
score is tensor([[ 2.0874, -2.4322, -0.9129,  ..., -0.0963,  1.3513, -0.6177],
        [ 1.9492, -1.3392, -0.3098,  ..., -0.6450, -0.2037, -0.1169],
        [ 1.0224, -1.5350, -0.7084,  ..., -0.0793, -0.0376,  0.1168],
        ...,
        [-0.3838, -0.2532,  0.8411,  ...,  2.5595,  0.4477, -0.3590],
        [ 1.0920,  0.4845,  0.7248,  ..., -0.8678,  0.3174, -0.7315],
        [-0.3707, -1.3360, -0.6462,  ..., -0.9208,  0.4862,  0.0177]])
output size is torch.Size([16, 10, 768])
score is tensor([[ 1.8192, -2.4388, -1.3585,  ..., -0.2315,  1.6503, -0.6870],
        [ 1.9856, -1.2579, -0.6497,  ..., -0.5458, -0.2931,  0.0679],
        [ 0.7866, -1.6967, -0.7640,  ..., -0.1282, -0.0576,  0.1798],
        ...,
        [-0.4298, -0.0833,  0.2765,  ...,  2.4889,  0.5125, -0.2009],
        [ 1.4766, -0.0783,  0.5334,  ..., -0.8563,  0.3350, -0.4659],
        [-0.2843, -1.5422, -0.8916,  ..., -1.1284,  0.5959, -0.0431]],
       grad_fn=<SelectBackward0>

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    Transformer Decoder Block (pure decoder structure)

    A single block of the Transformer decoder that includes:
    - Masked Multi-Head Self-Attention
    - Position-wise Feed-Forward Network
    - Layer Normalization and residual connections
    """
    def __init__(self, d_model, num_heads, d_ff):
        """
        Initialize the Transformer decoder block.

        Args:
            d_model (int): Dimensionality of the model embeddings.
            num_heads (int): Number of attention heads.
            d_ff (int): Dimensionality of the feed-forward network.
        """
        super().__init__()
        
        # Core modules
        self.self_attn = MultiheadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        
        # Final normalization after feed-forward
        self.final_norm = nn.LayerNorm(d_model)
        
        # Optional dropout for additional regularization
        # self.block_dropout = nn.Dropout(dropout_rate)

    def forward(self, x, attn_mask=None):
        """
        Forward pass for the Transformer decoder block.

        Args:
            x: Input tensor [batch_size, seq_len, d_model]
            attn_mask: Optional attention mask tensor

        Returns:
            output: Output tensor [batch_size, seq_len, d_model]
            attn_weights: Attention weights 
                          [batch_size, num_heads, seq_len, seq_len]
        """
        # Self-attention layer (with optional causal mask)
        attn_output, attn_weights = self.self_attn(x, attn_mask)
        
        # Position-wise feed-forward layer
        ff_output = self.feed_forward(attn_output)
        
        # Final normalization
        output = self.final_norm(ff_output)
        
        # Optional dropout
        # output = self.block_dropout(output)
        
        return output, attn_weights


In [ ]:
# Define test parameters
batch_size, seq_len, hidden_size = 16, 10, 768

# Create random input tensor simulating Transformer embeddings
x = torch.randn(batch_size, seq_len, hidden_size)

print(f"Input size: {x.size()}")
print(f"First sample (before decoder block):\n{x[0]}")

# Initialize the Transformer decoder block
# 12 heads, feed-forward expansion = 4 * hidden_size
decoder_block = TransformerDecoderBlock(hidden_size, 12, hidden_size * 4)

# Forward pass through the decoder block
output, attn_scores = decoder_block(x)

print(f"\nOutput size: {output.size()}")
print(f"First sample (after decoder block):\n{output[0]}")

# Inspect attention weights
print(f"\nAttention weights shape: {attn_scores.size()}")

intput size is torch.Size([16, 10, 768])
score is tensor([[-1.0234,  0.3199, -1.8247,  ..., -0.6495, -2.1413,  0.9001],
        [ 1.1926,  0.0670, -1.9403,  ...,  0.8865, -0.8397, -0.5059],
        [ 1.2522,  1.6009,  0.3749,  ..., -1.7742, -2.0079, -1.8397],
        ...,
        [-0.0038,  0.3279,  1.1320,  ..., -0.7547,  0.1903, -0.2759],
        [-1.2199,  0.7957, -0.8674,  ..., -2.1155,  1.5322, -2.2838],
        [ 0.7547,  0.0360, -0.8413,  ...,  2.4725,  2.0040,  0.1017]])
output size is torch.Size([16, 10, 768])
score is tensor([[ 1.4842,  0.7702, -1.0971,  ...,  0.8709, -1.2008, -0.2940],
        [ 1.3067,  0.7698, -0.8569,  ...,  0.3159, -1.1047, -0.5129],
        [ 1.7108,  0.7738, -0.6281,  ...,  0.8516, -0.1438,  0.0367],
        ...,
        [ 1.3382,  0.2951, -1.1412,  ...,  0.7016, -0.5304, -0.5005],
        [ 2.1465,  0.6518, -0.4792,  ...,  0.9102,  0.1186, -0.5564],
        [ 1.0797,  1.3547, -1.0620,  ...,  0.7938, -1.0190, -0.2776]],
       grad_fn=<SelectBackward0>

In [ ]:
import math
import torch
import torch.nn as nn

class TransformerDecoder(nn.Module):
    """
    Full Transformer Decoder Model (GPT-like Architecture)

    A stack of Transformer decoder blocks with causal masking for autoregressive modeling.
    This implementation supports:
      - Token and positional embeddings
      - Multiple decoder layers
      - Weight tying between embedding and output projection
      - Improved initialization and dropout regularization
    """
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Embedding layers
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_seq_len)
        
        # Stacked decoder blocks
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.final_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share weights between input embedding and output projection
        self.token_embed.weight = self.output_layer.weight
        
        # Embedding dropout
        self.embed_dropout = nn.Dropout(dropout_rate)
        
        # Initialize parameters
        self._init_weights()
    
    def _init_weights(self):
        """Improved weight initialization for stability and performance."""
        # Token embedding initialization
        nn.init.normal_(self.token_embed.weight, std=0.02)
        
        # Layer-wise initialization
        for layer in self.layers:
            # Attention projection layers
            nn.init.xavier_uniform_(layer.self_attn.query_proj.weight)
            nn.init.xavier_uniform_(layer.self_attn.key_proj.weight)
            nn.init.xavier_uniform_(layer.self_attn.val_proj.weight)
            nn.init.xavier_uniform_(layer.self_attn.out_proj.weight)
            
            # Feed-forward layers
            nn.init.kaiming_normal_(layer.feed_forward.linear1.weight)
            nn.init.kaiming_normal_(layer.feed_forward.linear2.weight)
    
    def create_causal_mask(self, seq_len):
        """
        Create a causal (lower-triangular) mask to prevent attending to future tokens.

        Returns:
            mask: Boolean tensor [seq_len, seq_len]
        """
        mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
        return mask
    
    def forward(self, input_ids):
        """
        Forward pass for the Transformer decoder.

        Args:
            input_ids: Input token IDs [batch_size, seq_len]

        Returns:
            logits: Prediction logits [batch_size, seq_len, vocab_size]
            all_attn_weights: List of attention weight tensors from each layer
                              [num_layers, batch_size, num_heads, seq_len, seq_len]
        """
        batch_size, seq_len = input_ids.shape
        
        # Token embeddings
        token_embeds = self.token_embed(input_ids)  # [batch, seq, d_model]
        token_embeds = token_embeds * math.sqrt(self.d_model)  # Scale embeddings
        
        # Add positional encoding and dropout
        embeddings = self.pos_encoder(token_embeds)
        embeddings = self.embed_dropout(embeddings)
        
        # Create causal mask (1 = allowed, 0 = blocked)
        causal_mask = self.create_causal_mask(seq_len).to(input_ids.device)
        causal_mask = causal_mask.unsqueeze(0)  # [1, seq_len, seq_len]
        
        # Pass through all decoder layers
        hidden_states = embeddings
        all_attn_weights = []
        for layer in self.layers:
            hidden_states, attn_weights = layer(hidden_states, causal_mask)
            all_attn_weights.append(attn_weights.detach())  # Save for visualization
        
        # Final normalization and output projection
        hidden_states = self.final_norm(hidden_states)
        logits = self.output_layer(hidden_states)
        
        return logits, all_attn_weights


In [ ]:
# Instantiate the Transformer decoder model
model = TransformerDecoder(
    vocab_size=500,      # vocabulary size
    d_model=256,         # embedding dimension
    num_layers=12,       # number of Transformer blocks
    num_heads=8,         # attention heads per block
    d_ff=256 * 4,        # feed-forward hidden size
    max_seq_len=128,     # maximum sequence length
    dropout_rate=0.1     # dropout for embeddings and layers
)

# Print model summary (optional)
print(model)

In [ ]:
# Create a batch of random token IDs to simulate input text
batch_size, seq_len = 4, 16
input_ids = torch.randint(0, 500, (batch_size, seq_len))  # [batch_size, seq_len]

# Forward pass through the model
logits, all_attn_weights = model(input_ids)

# Check output shapes
print(f"\nInput IDs shape: {input_ids.shape}")
print(f"Logits shape: {logits.shape}")  # Expected: [batch_size, seq_len, vocab_size]
print(f"Attention weights from first layer: {all_attn_weights[0].shape}")  # [batch, heads, seq, seq]

# Verify causal masking (no attention beyond current position)
print("\nCausal mask check (layer 1, head 0):")
print(all_attn_weights[0][0, 0])  # Attention matrix for one head

intput size is torch.Size([16, 10])
score is tensor([436, 446, 137, 394, 271, 393, 297, 235, 230, 463])
output size is torch.Size([16, 10, 500])
output is tensor([[ 0.2440, -1.1921, -0.5224,  ..., -0.3855, -0.1659,  0.3658],
        [ 0.2440, -1.1921, -0.5224,  ..., -0.3856, -0.1660,  0.3657],
        [ 0.2440, -1.1920, -0.5224,  ..., -0.3857, -0.1661,  0.3657],
        ...,
        [ 0.2439, -1.1918, -0.5223,  ..., -0.3860, -0.1665,  0.3653],
        [ 0.2439, -1.1918, -0.5223,  ..., -0.3860, -0.1666,  0.3653],
        [ 0.2439, -1.1918, -0.5223,  ..., -0.3861, -0.1667,  0.3652]],
       grad_fn=<SelectBackward0>)
